# 🗺️ View — Receita por Estado

Validação da view `vw_receita_por_estado` antes de mover para o Streamlit.

In [2]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

#formata todos os números float com 2 casas decimais na exibição do Jupyter.
pd.set_option('display.float_format', '{:.2f}'.format)

pedidos    = pd.read_csv("../dados/pedidos_limpo.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
clientes   = pd.read_csv("../dados/clientes_limpo.csv")
pagamentos = pd.read_csv("../dados/pagamentos_limpo.csv")

print("Dados carregados!")

Dados carregados!


## 🧪 Testando o código antes de criar a view

In [4]:
pedidos.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13


In [5]:
clientes.head(2)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP


In [6]:
pagamentos.head(2)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39


In [ ]:
 # ==========================================
 # 1. CONSOLIDANDO OS PAGAMENTOS POR PEDIDO
 # ==========================================
  
pagamentos_consolidados = (pagamentos.groupby('order_id', as_index=False)
        .agg(total_pago_pedido=('payment_value', 'sum')))

# ==========================================
# 2. Join 
# ==========================================
df = (pedidos.merge(
            clientes[
                ['customer_id', 'customer_state', 'customer_city']],
            on='customer_id',
            how='left').merge(pagamentos_consolidados,on='order_id',how='left'))

    # ==========================================
    # 3. FILTRANDO PEDIDOS ENTREGUES
    # ==========================================
df = df[df['order_status'] == 'delivered'].copy()

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,total_pago_pedido
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO,vianopolis,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,RN,sao goncalo do amarante,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,SP,santo andre,28.62


In [5]:
# ==========================================
    # 4. CRIANDO DIMENSÕES DE TEMPO
    # ==========================================
    # Ano da compra
df['ano'] = (df['order_purchase_timestamp'].dt.year)

    # Mês da compra
df['data_mes'] = (df['order_purchase_timestamp'].dt.to_period('M').astype(str))

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,total_pago_pedido,ano,data_mes
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,38.71,2017,2017-10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,141.46,2018,2018-07
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO,vianopolis,179.12,2018,2018-08
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,RN,sao goncalo do amarante,72.20,2017,2017-11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,SP,santo andre,28.62,2018,2018-02


In [10]:
  # ==========================================
    # 5. CRIANDO CÓDIGO GEOGRÁFICO DO ESTADO
    # ==========================================
    # Formato utilizado para representar os estados
    # brasileiros em mapas geográficos.
    # Exemplo: SP → BR-SP
df['state_geo'] = ('BR-' + df['customer_state'])
df.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,total_pago_pedido,ano,data_mes,dias_separacao,dias_transporte,dias_total,state_geo
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,38.71,2017,2017-10,2.00,6.00,8.00,BR-SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,141.46,2018,2018-07,1.00,12.00,13.00,BR-BA


In [16]:
  # ==========================================
    # 6. CALCULANDO PRAZOS DE ENTREGA
    # ==========================================
    # Tempo entre a compra e a entrega à transportadora
df['dias_separacao'] = (
    df['order_delivered_carrier_date'].dt.normalize()- df['order_purchase_timestamp'].dt.normalize()).dt.days

    # Tempo entre a entrega à transportadora e a entrega
    # ao cliente
df['dias_transporte'] = (
    df['order_delivered_customer_date'].dt.normalize()- df['order_delivered_carrier_date'].dt.normalize()).dt.days

    # Tempo total entre a compra e a entrega ao cliente
df['dias_total'] = (
    df['order_delivered_customer_date'].dt.normalize()- df['order_purchase_timestamp'].dt.normalize()).dt.days

df.head(2)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,total_pago_pedido,ano,data_mes,dias_separacao,dias_transporte,dias_total,state_geo
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,38.71,2017,2017-10,2.00,6.00,8.00,BR-SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,141.46,2018,2018-07,2.00,12.00,14.00,BR-BA


In [ ]:
# ==========================================
    # 7. AGRUPANDO OS DADOS
    # ==========================================
resultado = (
    df.groupby(
        ['customer_state', 'state_geo', 'ano', 'data_mes']
    )
    .agg(
        total_pedidos=('order_id', 'nunique'),
        total_clientes=('customer_id', 'nunique'),
        receita_total=('total_pago_pedido', 'sum'),
        soma_dias_separacao=('dias_separacao', 'sum'),
        soma_dias_transporte=('dias_transporte', 'sum'),
        soma_dias_total=('dias_total', 'sum'),
        total_pedidos_entregues=('order_id', 'nunique')
    )
    .reset_index()
)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,total_pago_pedido,ano,data_mes,dias_separacao,dias_transporte,dias_total,state_geo
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,38.71,2017,2017-10,2.00,6.00,8.00,BR-SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,141.46,2018,2018-07,1.00,12.00,13.00,BR-BA


In [13]:
    # ==========================================
    # 8. CALCULANDO INDICADORES DERIVADOS
    # ==========================================

    # Receita total
resultado['receita_total'] = (resultado['receita_total'].round(2))

    # Ticket médio por pedido
resultado['ticket_medio'] = (resultado['receita_total'] / resultado['total_pedidos']).round(2)

    # Prazo médio de separação
resultado['prazo_separacao_dias'] = (resultado['soma_dias_separacao'] / resultado['total_pedidos_entregues']).round(1)

    # Prazo médio de transporte
resultado['prazo_transporte_dias'] = (resultado['soma_dias_transporte'] / resultado['total_pedidos_entregues']).round(1)

    # Prazo médio total de entrega
resultado['prazo_total_dias'] = (resultado['soma_dias_total'] / resultado['total_pedidos_entregues']).round(1)

df.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,total_pago_pedido,ano,data_mes,dias_separacao,dias_transporte,dias_total,state_geo
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,38.71,2017,2017-10,2.00,6.00,8.00,BR-SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,141.46,2018,2018-07,1.00,12.00,13.00,BR-BA


In [19]:
from views.vw_receita_por_estado import get_receita_por_estado

df_estado = get_receita_por_estado(pedidos, clientes, pagamentos)
df_estado.head(2)

,customer_state,state_geo,ano,data_mes,total_pedidos,total_clientes,receita_total,soma_dias_separacao,soma_dias_transporte,soma_dias_total,total_pedidos_entregues,ticket_medio,prazo_separacao_dias,prazo_transporte_dias,prazo_total_dias
0,AC,BR-AC,2017,2017-01,2,2,723.15,6.00,33.00,40.00,2,361.58,3.00,16.50,20.00
1,AC,BR-AC,2017,2017-02,3,3,597.40,12.00,63.00,76.00,3,199.13,4.00,21.00,25.30
